# 7. Agent的高级用法3：结构化输出

结构化输出是Agent的核心功能之一，它允许Agent以特定、可预测的格式返回数据，而不是传统的自然
语言响应。通过结构化输出，开发者可以直接获得 Pydantic模型 、 JSON对象 或 数据类 等结构化数
据，这些数据能够被应用程序直接使用，无需复杂的解析过程。

## 7.1 模型 vs Agent的结构化输出对比

第06章已经介绍过结构化输出，当时的重点是与 `模型对象` 的绑定，这里与Agent的结构化输出做对
比：

| 对比项 | 模型：`model.with_structured_output(Schema)`（第六章） | Agent：`create_agent(..., response_format=Schema)` |
|---|---|---|
| 返回值 | Runnable，`invoke` 直接返回 Pydantic 对象或字典 | Agent，`invoke` 返回状态字典，结构化结果在 `result["structured_response"]`，`messages` 里是完整过程 |
| 过程 | 调用一次模型就结束 | 可以先调用工具收集信息，最后再给出结构化结果 |
| 工具 | 不会执行工具 | 能执行工具，工具和结构化输出在同一个循环里完成 |
| 怎样让模型按结构输出 | `method` 参数：`json_schema` / `function_calling` / `json_mode` | 策略：`ProviderStrategy` / `ToolStrategy` / 自动选择 |
| 默认怎么选 | 由模型集成决定：`ChatOpenAI` 默认 `json_schema`，`ChatDeepSeek` 默认 `function_calling` | 由 `create_agent` 根据模型档案（`model.profile`）自动选择 |
| 严格模式 | `ChatOpenAI` 的 `json_schema`：OpenAI SDK 自动开启 strict | `ProviderStrategy` 默认**不开启**，要写成 `ProviderStrategy(Schema, strict=True)` |
| 解析失败 | 抛出异常（`include_raw=True` 可以查看原始输出） | `ToolStrategy` 默认把错误反馈给模型，让它重试；`ProviderStrategy` 直接抛出异常 |
| 适合 | 一步就能完成的抽取、分类 | 需要先查资料、调用工具，再输出结构化结果的任务 |

两边的方式可以一一对应：模型的 `json_schema`（厂商原生结构化输出）相当于 Agent 的 `ProviderStrategy`；`function_calling`（把 schema 当成一个工具）相当于 `ToolStrategy`。

最大的区别在于**能不能先用工具**。下面让两者回答同一个问题：查员工张三的联系方式。这些信息只在公司通讯录里，要调用工具才能查到。先准备模型、schema 和工具（不请求模型）：

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
)

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

@tool
def get_employee_contact(name: str) -> str:
    """根据员工姓名，查询公司通讯录中的邮箱和手机号"""
    directory = {"张三": "邮箱 zhangsan@atguigu.com，手机 13800001111"}
    return directory.get(name, f"通讯录里没有 {name}")

同一个问题分别交给模型和 Agent（会请求模型三次左右：模型一次，Agent 两次以上）。

Agent 这边加了一句系统提示词，要求"查到信息后调用 `ContactInfo` 提交结果"。原因是：测试时发现，不加这句话，这个中转模型在调用过通讯录工具后，有时会**无视"必须调用工具"的要求**，直接用文字回答，导致拿不到结构化结果（详见 7.4.4）：

In [8]:
question = "帮我查一下员工张三的联系方式"

# ① 模型：只调用一次，查不了通讯录
model_result = model.with_structured_output(ContactInfo).invoke(question)
print("模型的结果:", repr(model_result))
print()

# ② Agent：先调用工具查通讯录，再给出结构化结果
contact_agent = create_agent(
    model=model,
    tools=[get_employee_contact],
    response_format=ToolStrategy(ContactInfo),
    system_prompt="先用 get_employee_contact 查询通讯录；查到信息后，必须调用 ContactInfo 工具提交结果，不要直接用文字回答。",
)
agent_result = contact_agent.invoke({"messages": [{"role": "user", "content": question}]})
for msg in agent_result["messages"]:
    body = f"调用 {msg.tool_calls[0]['name']}{msg.tool_calls[0]['args']}" if getattr(msg, "tool_calls", None) else msg.text
    print(f"    {type(msg).__name__:<12} {body}")
print("Agent 的结果:", repr(agent_result["structured_response"]))

模型的结果: ContactInfo(name='张三', email='', phone='')



    HumanMessage 帮我查一下员工张三的联系方式
    AIMessage    调用 get_employee_contact{'name': '张三'}
    ToolMessage  邮箱 zhangsan@atguigu.com，手机 13800001111
    AIMessage    调用 ContactInfo{'email': 'zhangsan@atguigu.com', 'name': '张三', 'phone': '13800001111'}
    ToolMessage  Returning structured response: name='张三' email='zhangsan@atguigu.com' phone='13800001111'
Agent 的结果: ContactInfo(name='张三', email='zhangsan@atguigu.com', phone='13800001111')


对比结果：

- **模型**只调用了一次，查不了通讯录，只能把邮箱和手机号留空（`ChatOpenAI` 的 `json_schema` 是 strict 模式，每个字段都必须有值，于是给了空字符串）
- **Agent** 先调用 `get_employee_contact` 查到信息，再"调用" `ContactInfo` 提交结果，拿到了完整的联系方式

这就是两者最大的区别：模型的结构化输出只能利用提问里已有的信息；Agent 可以先用工具把信息找齐，再按结构交出结果。

## 7.2 结构化输出的4种策略

LangChain的create_agent()函数自动处理结构化输出的全过程。用户只需通过 “response_format”参数 设置期望的输出模式（Schema）。

当模型生成结构化数据时，系统会自动捕获、验证并将结果存储在Agent状态的 structured_response键中。

```python
create_agent(
  *,
  response_format: 
    ResponseFormat[ResponseT] | 
    type[ResponseT] | 
    dict[str, Any] | 
    None = None,
  state_schema: type[AgentState[ResponseT]] | None = None,
  context_schema: type[ContextT] | None = None,
  checkpointer: Checkpointer | None = None,
  store: BaseStore | None = None,
) -> CompiledStateGraph[AgentState[ResponseT], ContextT, InputAgentState, OutputAgentState[ResponseT]]
```

response_format: 结构化响应的可选配置。可以是 ToolStrategy、ProviderStrategy 或 Pydantic 模型类。如果提供，agent将在对话流程中处理结构化输出。Raw schemas将根据模型能力被包装在适当的策略中。

create_agent函数中的response_format参数支持四种不同的策略设置方式：

### 7.2.1 ① ProviderStrategy
使用模型提供商的 原生结构化输出功能 实现结构化输出。

* 原理：利用大模型厂商（Provider）原生提供的 Structured Outputs（结构化输出）或 JSON Mode API。
* 做法：直接在底层 API 请求中给大模型套上枷锁（例如 OpenAI 新版的 response_format: {"type": "json_schema"}）。这是在模型生成 Token 的阶段做出的硬性概率限制，强制它吐出符合 Schema 的结果。
* 优势与场景：这是一种“直捣黄龙”的方案，更加直接、严格、稳定。适用于现代主流模型（如 GPT-4o、Claude 3.5 等）。




In [1]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.agents.structured_output import ProviderStrategy
from rich import print as rprint
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel, Field
from langchain.messages import SystemMessage, HumanMessage

load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

# 2.Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3. agent初始化
agent = create_agent(
    model=model,
    response_format=ProviderStrategy(ContactInfo)
)

# 4.调用
response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================

{"email":"shkstart@atguigu.com","name":"小明","phone":"12345678912"}


In [2]:
rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='ef0cfd56-f5a6-47b5-b3bf-60784b6a004e'
        ),
        AIMessage(
            content='{"email":"shkstart@atguigu.com","name":"小明","phone":"12345678912"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 31,
                    'prompt_tokens': 4536,
                    'total_tokens': 4567,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna',
                'system_fingerprint': None,
                'id': 'resp_04f991e8bb827f76016aba21b1553087d2a8e12ccec8713365',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a0e713-9149-76b3-855f-8f60ff2b4aa0-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4536,
                'output_tokens': 31,
                'total_tokens': 4567,
                'input_token_details': {},
                'output_token_details': {}
            }
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

注意两点：

- `AIMessage.content` 是一段 JSON 文本，`additional_kwargs` 里的 `parsed` 是 `None`。这和第六章的 `with_structured_output` 不同：这里不是 OpenAI SDK 解析的，而是 `create_agent` 自己用 `json.loads` 读出 JSON，再交给 Pydantic 创建 `ContactInfo`
- `ProviderStrategy` 默认**没有开启** strict（见 7.1 的对比表），需要由服务端严格保证格式时，写成 `ProviderStrategy(ContactInfo, strict=True)`

### 7.2.2 ② ToolStrategy
对于不支持原生结构化输出的模型，LangChain采用“ToolStrategy”工具调用的方式实现结构化输出。

此策略兼容绝大多数 支持工具调用 的现代模型，其核心原理是动态创建一个" 虚拟工具 "，该工具的输
入参数对应着期望的数据结构。

当模型需要生成最终答案时，系统会引导模型 "调用"这个虚拟工具 ，从而间接产生符合要求的结构化数
据。

In [3]:
import os
from httpx2 import query
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from rich import print as rprint
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel, Field
from langchain.messages import SystemMessage, HumanMessage

load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

# 2.Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3.工具的定义（根据需要定义）
@tool
def search_tool(query: str) -> str:
    """
    这是一个搜索引擎。当大模型发现给定的上下文里缺少必要的联系人信息，需要去互联网上查询时，才会调用这个工具。
    """
    return f"搜索结果: 未找到关于 '{query}' 的更多额外信息。"


# 4. agent初始化
agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)

# 4.调用
response = agent.invoke({
    "messages": [
        HumanMessage("联系人信息: John Doe,john@atguigu.com, (010) 56253825")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

联系人信息: John Doe,john@atguigu.com, (010) 56253825
================================== Ai Message ==================================

[{'arguments': '{"name":"John Doe","email":"john@atguigu.com","phone":"(010) 56253825"}', 'call_id': 'call_zBcuqHqxZV7ty9Uu9TOlQ7z5', 'name': 'ContactInfo', 'type': 'function_call', 'id': 'fc_02b4eca406cc39fe016aba27347f6c87d0b276692b031343f8', 'status': 'completed', 'internal_chat_message_metadata_passthrough': {'create_time': 1790584619.82889, 'turn_id': '01a0e728-fcb0-7562-aaa3-6cd9c55902bb'}, 'metadata': {'turn_id': '01a0e728-fcb0-7562-aaa3-6cd9c55902bb'}}]
Tool Calls:
  ContactInfo (call_zBcuqHqxZV7ty9Uu9TOlQ7z5)
 Call ID: call_zBcuqHqxZV7ty9Uu9TOlQ7z5
  Args:
    name: John Doe
    email: john@atguigu.com
    phone: (010) 56253825
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='John 

In [4]:
rprint(response)

{
    'messages': [
        HumanMessage(
            content='联系人信息: John Doe,john@atguigu.com, (010) 56253825',
            additional_kwargs={},
            response_metadata={},
            id='d4233004-eb4f-4296-b3f6-69d8512b3dfe'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"name":"John Doe","email":"john@atguigu.com","phone":"(010) 56253825"}',
                    'call_id': 'call_zBcuqHqxZV7ty9Uu9TOlQ7z5',
                    'name': 'ContactInfo',
                    'type': 'function_call',
                    'id': 'fc_02b4eca406cc39fe016aba27347f6c87d0b276692b031343f8',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'create_time': 1790584619.82889,
                        'turn_id': '01a0e728-fcb0-7562-aaa3-6cd9c55902bb'
                    },
                    'metadata': {'turn_id': '01a0e728-fcb0-7562-aaa3-6cd9c55902bb'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_02b4eca406cc39fe016aba272b626487d09b3c14f64398f5e7',
                'created_at': 1790584619.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_02b4eca406cc39fe016aba272b626487d09b3c14f64398f5e7',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': 'John Doe', 'email': 'john@atguigu.com', 'phone': '(010) 56253825'},
                    'id': 'call_zBcuqHqxZV7ty9Uu9TOlQ7z5',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 157,
                'output_tokens': 37,
                'total_tokens': 194,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='John Doe' email='john@atguigu.com' phone='(010) 
56253825'",
            name='ContactInfo',
            id='66bd719d-fdd8-4492-b1e9-a48bb04c33c5',
            tool_call_id='call_zBcuqHqxZV7ty9Uu9TOlQ7z5'
        )
    ],
    'structured_response': ContactInfo(name='John Doe', email='john@atguigu.com', phone='(010) 56253825')
}

和 7.2.1 相比多了两条消息：模型"调用"了一个名叫 `ContactInfo` 的工具来提交结果，参数就是结构化数据；Agent 解析之后补上一条 `ToolMessage`（`Returning structured response: ...`），然后结束运行。这个工具就是 `ToolStrategy` 根据 schema 生成的"虚拟工具"。

另外，这个单元格里的 `from httpx2 import query` 没有用到，可能是编辑器自动补全时加上的，可以删掉；`search_tool` 定义了，但没有传给 `create_agent` 的 `tools`，所以这次 Agent 其实没有可用的业务工具。

### 7.2.3 ③ AutoStrategy：自动选择策略

`AutoStrategy(ContactInfo)` 是一个自动选择策略的配置对象。运行时，`create_agent` 根据模型能力选择 `ProviderStrategy` 或 `ToolStrategy`。

源码中的 `ResponseFormat` 将这三种策略类型合在一起：

```python
ResponseFormat = ToolStrategy[SchemaT] | ProviderStrategy[SchemaT] | AutoStrategy[SchemaT]
```

直接传入 `ContactInfo` 类时，也会自动包装成 `AutoStrategy(ContactInfo)`。这里的 `SchemaT` 是类型变量，`ToolStrategy[SchemaT]` 等是泛型类型写法；具体解释见 7.3。

In [5]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.agents.structured_output import ProviderStrategy, AutoStrategy, ToolStrategy
from rich import print as rprint
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel, Field
from langchain.messages import SystemMessage, HumanMessage

load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

# 2.Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3. agent初始化
agent = create_agent(
    model=model,
    response_format=AutoStrategy(ContactInfo)
)

# 4.调用
response = agent.invoke({
    "messages": [
        HumanMessage("联系人信息: John Doe,john@atguigu.com, (010) 56253825")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

联系人信息: John Doe,john@atguigu.com, (010) 56253825
================================== Ai Message ==================================

{"email":"john@atguigu.com","name":"John Doe","phone":"(010) 56253825"}


In [6]:
rprint(response)

{
    'messages': [
        HumanMessage(
            content='联系人信息: John Doe,john@atguigu.com, (010) 56253825',
            additional_kwargs={},
            response_metadata={},
            id='6e546259-7ea9-4683-909b-7fe68ed27d04'
        ),
        AIMessage(
            content='{"email":"john@atguigu.com","name":"John Doe","phone":"(010) 56253825"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 32,
                    'prompt_tokens': 4523,
                    'total_tokens': 4555,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna',
                'system_fingerprint': None,
                'id': 'resp_09eb627556994af2016aba286a87d087d2a8ed5b6fbca5b302',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a0e72d-cc32-7093-81b1-2b1163444281-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4523,
                'output_tokens': 32,
                'total_tokens': 4555,
                'input_token_details': {},
                'output_token_details': {}
            }
        )
    ],
    'structured_response': ContactInfo(name='John Doe', email='john@atguigu.com', phone='(010) 56253825')
}

输出和 7.2.1 完全一样：只有一条带 JSON 文本的 `AIMessage`，没有工具调用。说明 `AutoStrategy` 选择了 `ProviderStrategy`，因为 `gpt-6-luna` 的模型档案里声明支持结构化输出。它是怎么判断的、什么时候会判断错，见 7.4.2。

### 7.2.4 直接传 Pydantic 类

`response_format` 也接受 `type[ResponseT]`，所以可以直接传入 `ContactInfo` 类本身：

```python
response_format=ContactInfo
```

当 `ResponseT` 对应 `ContactInfo` 时，`type[ResponseT]` 表示这里接收 `ContactInfo` 类对象；成功解析后，`structured_response` 才是 `ContactInfo` 实例。类、策略对象和实例的区别见 7.3。

In [7]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.agents.structured_output import ProviderStrategy, AutoStrategy, ToolStrategy
from rich import print as rprint
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel, Field
from langchain.messages import SystemMessage, HumanMessage

load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

# 2.Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3. agent初始化
agent = create_agent(
    model=model,
    response_format=ContactInfo
)

# 4.调用
response = agent.invoke({
    "messages": [
        HumanMessage("联系人信息: John Doe,john@atguigu.com, (010) 56253825")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

联系人信息: John Doe,john@atguigu.com, (010) 56253825
================================== Ai Message ==================================

{"email":"john@atguigu.com","name":"John Doe","phone":"(010) 56253825"}


直接传 Pydantic 类时，`create_agent` 会自动把它包装成 `AutoStrategy(ContactInfo)`（源码：`AutoStrategy(schema=response_format)`），所以结果和 7.2.3 一样。`response_format` 一共能传哪些值，见 7.3。

### 7.2.5 ④ None
默认配置，表示不以结构化输出，以 自然语言 响应用户问题。

总结：

**在实际大模型Agent开发场景中，如果使用到了结构化输出，推荐使用 “ToolStrategy”策略 ，所以后续
重点介绍这种策略方式结构化输出。**

## 7.3 response_format 的取值：从 Python 泛型读懂类型注解

`create_agent` 的参数注解中有这样一行：

```python
response_format: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None = None
```

理解这行代码，需要先区分**类型变量、类本身、策略对象和结果实例**。下面统一用前文的 Pydantic 模型 `ContactInfo` 来说明。

### 7.3.1 先看懂基本符号

| 写法 | 含义 | 例子 |
|---|---|---|
| `A \| B` | 允许 A 或 B 两种类型的值 | `str \| None`：字符串或 `None` |
| `= None` | 参数的默认值是不做配置 | 不传 `response_format` 时，它就是 `None` |
| `dict[str, Any]` | 键是字符串，值的类型不限 | 这里允许传入 JSON Schema 字典 |
| `type[X]` | 保存 X 类本身或其子类的类对象 | `schema: type[ContactInfo] = ContactInfo` |
| `X[Y]` | 给泛型 X 指定类型参数 Y；具体含义由 X 定义 | `list[int]` 的元素是整数；`ToolStrategy[ContactInfo]` 对应的结果类型是 `ContactInfo` |

不要把所有 `X[Y]` 都理解成“容器 X 里装着 Y”。列表用类型参数描述元素，而策略类用类型参数描述它对应的结构化结果。

### 7.3.2 泛型：同一套代码，保留不同类型之间的联系

先从列表开始：

```python
numbers: list[int] = [1, 2, 3]
names: list[str] = ["张三", "李四"]
```

`list` 是同一种列表，方括号说明元素类型。写成 `list[T]` 时，元素类型暂时用 `T` 表示，使用时再确定。

`T` 可以通过 `TypeVar` 定义。下面这个函数返回列表的第一个元素：

```python
from typing import TypeVar

T = TypeVar("T")

def first(items: list[T]) -> T:
    return items[0]

number = first([1, 2, 3])       # T 对应 int，返回值类型也是 int
name = first(["张三", "李四"])  # T 对应 str，返回值类型也是 str
```

函数的输入和返回值使用同一个 `T`，是在表达：“返回值的类型与输入列表的元素类型一致。”类型检查工具可以据此推断 `number` 是整数、`name` 是字符串。

如果把两处都写成 `Any`，虽然也能运行，但就没有表达这种输入与输出的对应关系。

### 7.3.3 ResponseT 和 SchemaT：两个分别定义的类型变量

本项目安装的 LangChain 源码分别定义了：

```python
# langchain/agents/middleware/types.py
ResponseT = TypeVar("ResponseT", default=Any)

# langchain/agents/structured_output.py
SchemaT = TypeVar("SchemaT")
```

| 类型变量 | 使用位置 | 在这里表示什么 |
|---|---|---|
| `SchemaT` | `ToolStrategy`、`ProviderStrategy`、`AutoStrategy` 等策略类 | 策略所对应的结构化数据类型 |
| `ResponseT` | `create_agent`、`AgentState` 等 | Agent 最终结构化结果的数据类型 |

它们是两个不同的 `TypeVar` 对象，但作用相似：都用来表达尚未确定的类型。名字没有 Python 内置的特殊含义，作者也可以取名为 `T`、`ResultT`。`ResponseT` 的 `default=Any` 为未指定的类型参数提供默认值。

对于本笔记的联系方式例子，策略里的 `SchemaT` 和 Agent 里的 `ResponseT` 最终都可以对应到 `ContactInfo`。这不是修改类型变量的值，而是类型检查工具根据传入的参数建立类型对应关系。

### 7.3.4 type[SchemaT]：接收的是类本身

当 `SchemaT` 对应 `ContactInfo` 时，`type[SchemaT]` 就对应 `type[ContactInfo]`。注意类与实例的区别：

```python
# 保存类本身：描述字段结构，还没有具体的联系方式
schema: type[ContactInfo] = ContactInfo

# 创建实例：已经有一条具体的联系方式
contact: ContactInfo = ContactInfo(
    name="张三",
    email="zhangsan@example.com",
    phone="13800001111",
)
```

传给 `response_format` 的 schema 是 `ContactInfo` 类本身；成功解析后得到的 `structured_response` 才是该类的实例。

“`int` 的类型是 `type[int]`”需要限定在**类型注解**的语境下理解：

```python
number: int = 123
number_class: type[int] = int

print(type(123))  # <class 'int'>
print(type(int))  # <class 'type'>
```

`type[int]` 是描述“这里保存的是 int 类对象”的注解；运行时调用 `type(int)` 得到的结果是 `type`。

### 7.3.5 ToolStrategy[SchemaT]：结果类型为 SchemaT 的工具策略

`ToolStrategy` 是一个泛型类。为了看清类型关系，可以把源码简化为：

```python
from typing import Generic, TypeVar

SchemaT = TypeVar("SchemaT")

class ToolStrategy(Generic[SchemaT]):
    def __init__(self, schema: type[SchemaT]):
        self.schema = schema
```

> 这段代码只用于说明类型关系，不要执行它来覆盖导入的 LangChain `ToolStrategy`。实际类还支持联合类型、JSON Schema 字典、错误处理等功能。

`Generic[SchemaT]` 声明这个类接收一个类型参数。将 `SchemaT` 对应到 `ContactInfo` 后：

- `ToolStrategy[ContactInfo]`：针对 `ContactInfo` 结果类型的策略对象的类型。
- 初始化参数 `schema: type[ContactInfo]`：接收 `ContactInfo` 类本身。

实际代码这样写：

```python
strategy: ToolStrategy[ContactInfo] = ToolStrategy(schema=ContactInfo)
```

冒号左侧的变量 `strategy` 保存一个策略对象；冒号右侧的 `ToolStrategy[ContactInfo]` 是类型注解；等号右侧用圆括号创建策略对象，并把 `ContactInfo` 类传给它。

| 值 | 它是什么 | 对应的类型注解 |
|---|---|---|
| `ContactInfo` | 描述联系方式字段的类对象 | `type[ContactInfo]` |
| `ToolStrategy(ContactInfo)` | 根据该类配置的工具策略对象 | `ToolStrategy[ContactInfo]` |
| `ContactInfo(name=..., email=..., phone=...)` | 一条具体的联系方式实例 | `ContactInfo` |

方括号并不只允许出现在类型注解中。下面这种“先显式指定类型参数，再创建对象”的写法也合法：

```python
strategy = ToolStrategy[ContactInfo](schema=ContactInfo)
```

日常使用 `ToolStrategy(ContactInfo)` 通常就足够了。运行时 `type(strategy)` 仍然是 `ToolStrategy`；方括号指定了更具体的类型关系，没有把策略对象变成 `ContactInfo` 实例。

### 7.3.6 ResponseFormat[ResponseT]：把策略类型与结果类型接起来

`ResponseFormat` 是类型别名，为三种策略类型起了一个共同的名称：

```python
ResponseFormat = (
    ToolStrategy[SchemaT]
    | ProviderStrategy[SchemaT]
    | AutoStrategy[SchemaT]
)
```

所以 `ResponseFormat[ResponseT]` 可以展开理解为：

```python
ToolStrategy[ResponseT] | ProviderStrategy[ResponseT] | AutoStrategy[ResponseT]
```

这一步把类型别名中的参数 `SchemaT` 对应到外面的 `ResponseT`，让策略对应的数据类型与 Agent 最终结果的数据类型一致。

当具体类型是 `ContactInfo` 时，参数注解可以理解为：

```python
# 原始的通用注解
response_format: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None

# 用 ContactInfo 理解这次调用的类型关系
response_format: ResponseFormat[ContactInfo] | type[ContactInfo] | dict[str, Any] | None
```

这只是帮助阅读类型注解；不需要去修改 LangChain 的函数签名。

### 7.3.7 将 ResponseT 对应到 Pydantic 模型，有什么意义？

例如这样配置 Agent：

```python
agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo),
)
```

通用的类型关系就可以具体理解为：

```text
ToolStrategy[SchemaT]      → ToolStrategy[ContactInfo]
ResponseFormat[ResponseT] → ResponseFormat[ContactInfo]
AgentState[ResponseT]     → AgentState[ContactInfo]
structured_response      → ContactInfo 实例
```

无需修改库源码，也无需赋值 `ResponseT = ContactInfo`。直接传 `response_format=ContactInfo` 时也表达同样的结果类型，只是由 Agent 自动选择策略。

这里要区分两件事：

1. **泛型的作用**：表达“传入 ContactInfo，结构化结果也是 ContactInfo”的类型联系，为编辑器提供类型检查和自动补全信息。
2. **Pydantic 的作用**：在实际运行时定义字段、校验数据，并由 LangChain 将解析后的数据转换成 `ContactInfo` 实例。

Python 本身不会因为写了 `ResponseT` 或 `ToolStrategy[ContactInfo]` 就自动验证模型输出。运行时的数据校验来自 LangChain 调用 Pydantic 的逻辑。

结构化输出成功后，就可以按对象字段使用结果：

```python
result = agent.invoke({
    "messages": [{"role": "user", "content": "张三的邮箱是 zhangsan@example.com，手机是 13800001111"}]
})

contact = result["structured_response"]
print(contact.name)
print(contact.email)
```

后面的 7.4.4 还会说明：实际业务中要检查结构化结果是否成功产生，类型注解本身不能保证模型一定提交结果。

参考：[Python 泛型](https://docs.python.org/3/library/typing.html#generics)、[类对象的类型注解](https://docs.python.org/3/library/typing.html#the-type-of-class-objects)、[LangChain Agent 结构化输出](https://docs.langchain.com/oss/python/langchain/structured-output)。

下面的代码可以单独运行，观察类型变量、类、策略对象和 Pydantic 实例，**不请求大模型**：

In [1]:
import inspect
from typing import get_args
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ResponseFormat, SchemaT, ToolStrategy
from langchain.agents.middleware.types import ResponseT

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

print("参数的类型注解:", inspect.signature(create_agent).parameters["response_format"].annotation)
print("ResponseT:", ResponseT, "| 它是:", type(ResponseT).__name__)
print("SchemaT:", SchemaT, "| 它是:", type(SchemaT).__name__)
print("它们是同一个类型变量吗？", ResponseT is SchemaT)

print("\nResponseFormat[ContactInfo] 展开后包含:")
for strategy_type in get_args(ResponseFormat[ContactInfo]):
    print("   ", strategy_type)

schema: type[ContactInfo] = ContactInfo
strategy: ToolStrategy[ContactInfo] = ToolStrategy(schema=schema)
explicit_strategy = ToolStrategy[ContactInfo](schema=ContactInfo)
contact: ContactInfo = ContactInfo(
    name="张三", email="zhangsan@example.com", phone="13800001111"
)

print("\n类对象:", schema)
print("策略对象的运行时类型:", type(strategy).__name__)
print("策略里的 schema 是 ContactInfo 吗？", strategy.schema is ContactInfo)
print("显式指定泛型参数后，schema 仍是 ContactInfo 吗？", explicit_strategy.schema is ContactInfo)
print("结果实例的运行时类型:", type(contact).__name__)
print("访问实例字段:", contact.name, contact.email)

print("\ntype(123):", type(123))
print("type(int):", type(int))

参数的类型注解: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None
ResponseT: ~ResponseT | 它是: TypeVar
SchemaT: ~SchemaT | 它是: TypeVar
它们是同一个类型变量吗？ False

ResponseFormat[ContactInfo] 展开后包含:
    langchain.agents.structured_output.ToolStrategy[__main__.ContactInfo]
    langchain.agents.structured_output.ProviderStrategy[__main__.ContactInfo]
    langchain.agents.structured_output.AutoStrategy[__main__.ContactInfo]

类对象: <class '__main__.ContactInfo'>
策略对象的运行时类型: ToolStrategy
策略里的 schema 是 ContactInfo 吗？ True
显式指定泛型参数后，schema 仍是 ContactInfo 吗？ True
结果实例的运行时类型: ContactInfo
访问实例字段: 张三 zhangsan@example.com

type(123): <class 'int'>
type(int): <class 'type'>


`~ResponseT`、`~SchemaT` 前面的波浪号，是 Python 打印类型变量时的表示方式。上面的 `False` 说明它们是两个分别定义的对象；展开后的策略类型都使用了具体的 `ContactInfo`。

### 7.3.8 四种取值，以及 structured_response 会是什么

| 取值 | 写法举例 | `structured_response` 的类型 | 会不会校验 |
|---|---|---|---|
| 策略对象 `ResponseFormat[ResponseT]` | `ToolStrategy(ContactInfo)`、`ProviderStrategy(ContactInfo)`、`AutoStrategy(ContactInfo)` | 由策略里的 schema 决定，同下 | 同下 |
| 类 `type[ResponseT]` | Pydantic 类 | 该类的实例 | 会 |
| | dataclass | 该 dataclass 的实例 | 会 |
| | TypedDict | 字典 | 会 |
| JSON Schema 字典 `dict[str, Any]` | `{"title": "ContactInfo", "type": "object", ...}` | 字典 | **不会**，模型给什么就返回什么 |
| `None` | 不传 | 返回值里没有结构化结果 | |

两点注意：

1. 直接传类或字典时，`create_agent` 会自动把它包装成 `AutoStrategy`，所以 7.2.3 和 7.2.4 的效果完全一样
2. TypedDict 必须从 `typing_extensions` 导入。`create_agent` 内部用 Pydantic 处理 schema，在 Python 3.12 以下用 `typing.TypedDict` 会直接报错。第六章 04 笔记用 `with_structured_output` 时没有这个限制

## 7.4 五种写法的区别，以及实际业务中用哪一种

### 7.4.1 区别

| 写法 | 实际使用的策略 | 模型怎样交出结果 | 优点 | 缺点 |
|---|---|---|---|---|
| `ProviderStrategy(S)` | 厂商原生结构化输出 | 直接输出一段 JSON 文本 | 最直接；开启 `strict=True` 后由服务端保证格式 | 只有部分厂商和模型支持；默认不开 strict；解析失败直接抛异常，不会重试 |
| `ToolStrategy(S)` | 工具调用 | "调用"一个和 schema 同名的虚拟工具 | 支持工具调用的模型基本都能用；校验失败时自动把错误反馈给模型、让它重试；支持 `Union` 多选一 | 消息里多一轮工具调用；默认不开 strict，靠重试兜底 |
| `AutoStrategy(S)` | 根据模型档案二选一 | 取决于选中的策略 | 不用自己判断 | 完全依赖模型档案，档案不准就会选错；换个模型，行为可能悄悄改变 |
| `S`（直接传类） | 等同 `AutoStrategy(S)` | 同上 | 写法最短 | 同上 |
| `None` | 不做结构化输出 | 自然语言 | | 需要自己解析文本 |

7.2 的输出正好印证了这张表：`ProviderStrategy`、`AutoStrategy`、直接传类这三次，返回的消息都是"用户消息 + 一条带 JSON 文本的 AI 消息"；只有 `ToolStrategy` 多了"工具调用 + 工具消息"。

### 7.4.2 自动选择的依据，以及它为什么可能选错

自动选择只看一件事：模型档案（`model.profile`）里有没有声明支持结构化输出。没有档案的模型，再按模型名匹配一个内置列表（如 `gpt-4o`、`gpt-5`）。下面对三个模型分别做判断（调用的是内部函数，仅用于观察，不请求模型）：

In [10]:
from langchain_deepseek import ChatDeepSeek
from langchain.agents.factory import _supports_provider_strategy   # 内部函数，仅用于观察

candidates = {
    "gpt-6-luna（本笔记用的模型）": model,
    "deepseek-v4-flash": ChatDeepSeek(model="deepseek-v4-flash", api_key="demo"),
    "qwen-plus（通过兼容接口接入）": ChatOpenAI(model="qwen-plus", api_key="demo", base_url="https://example.com/v1"),
}
for label, candidate in candidates.items():
    declared = (candidate.profile or {}).get("structured_output")
    chosen = "ProviderStrategy" if _supports_provider_strategy(candidate) else "ToolStrategy"
    print(f"{label}：档案声明支持结构化输出 = {declared}，自动选择 → {chosen}")

gpt-6-luna（本笔记用的模型）：档案声明支持结构化输出 = True，自动选择 → ProviderStrategy
deepseek-v4-flash：档案声明支持结构化输出 = True，自动选择 → ProviderStrategy
qwen-plus（通过兼容接口接入）：档案声明支持结构化输出 = None，自动选择 → ToolStrategy


- `gpt-6-luna` 的档案声明支持，所以 7.2.3、7.2.4 自动选了 `ProviderStrategy`
- DeepSeek 的档案也声明支持，自动选择同样是 `ProviderStrategy`，但 DeepSeek 的接口并不接受 `json_schema` 格式，01 笔记 2.5.4 就因此收到了 400 错误。这就是"档案不准，选错策略"
- `qwen-plus` 通过 OpenAI 兼容接口接入，LangChain 没有它的档案，名字也不在内置列表里，于是选了 `ToolStrategy`

也就是说，同一段"直接传类"的代码，换一个模型名，使用的策略就可能变掉。

### 7.4.3 ToolStrategy 的自动纠错

真实模型很少出错，不方便演示。下面用一个"按剧本依次回复"的假模型（`GenericFakeChatModel`，LangChain 自带的测试工具），模拟"第一次漏掉了 `phone`，看到错误信息后补全"的过程（不请求真实模型）：

In [11]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

class ScriptedModel(GenericFakeChatModel):
    """按剧本依次返回消息的假模型，只用来演示流程"""
    def bind_tools(self, tools, **kwargs):
        return self

scripted_model = ScriptedModel(messages=iter([
    # 第 1 次：漏掉了 phone
    AIMessage(content="", tool_calls=[{"name": "ContactInfo", "args": {"name": "小明", "email": "shkstart@atguigu.com"}, "id": "call_1", "type": "tool_call"}]),
    # 第 2 次：看到错误信息后补全
    AIMessage(content="", tool_calls=[{"name": "ContactInfo", "args": {"name": "小明", "email": "shkstart@atguigu.com", "phone": "12345678912"}, "id": "call_2", "type": "tool_call"}]),
]))

retry_agent = create_agent(model=scripted_model, response_format=ToolStrategy(ContactInfo))
result = retry_agent.invoke({"messages": [{"role": "user", "content": "从这段话中抽取联系方式：小明的邮箱是 shkstart@atguigu.com，手机号 12345678912"}]})

for msg in result["messages"]:
    body = msg.tool_calls[0]["args"] if getattr(msg, "tool_calls", None) else msg.text
    print(f"{type(msg).__name__:<12} {body}\n")
print("structured_response:", repr(result["structured_response"]))

HumanMessage 从这段话中抽取联系方式：小明的邮箱是 shkstart@atguigu.com，手机号 12345678912

AIMessage    {'name': '小明', 'email': 'shkstart@atguigu.com'}

ToolMessage  Error: Failed to parse structured output for tool 'ContactInfo': Failed to parse data to ContactInfo: 1 validation error for ContactInfo
phone
  Field required [type=missing, input_value={'name': '小明', 'email... 'shkstart@atguigu.com'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing.
 Please fix your mistakes.

AIMessage    {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'}

ToolMessage  Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'

structured_response: ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')


第一次提交缺少 `phone`，Pydantic 校验失败。Agent 没有抛出异常，而是把错误信息写进一条 `ToolMessage`（结尾是 `Please fix your mistakes.`）交回模型；模型第二次补全后，才得到 `structured_response`。这是 `ToolStrategy` 的 `handle_errors` 参数控制的，默认就是开启的。

### 7.4.4 实际业务中用哪一种？

笔记前面的结论是推荐 `ToolStrategy`。结合前面的实验，原因可以归纳为四点：

1. **兼容性最好**：它只要求模型支持工具调用，而这本来就是 Agent 必需的能力。国内常用的 DeepSeek 等模型不支持 `json_schema` 原生结构化输出，只能用它
2. **行为可预期**：显式写出策略，不依赖模型档案的自动判断。直接传类（自动选择）时，换一个模型，策略就可能悄悄变掉（7.4.2）
3. **自带纠错**：模型给错了会收到错误信息并重新提交（7.4.3）。业务里"偶尔格式不对"很常见，自动重试能明显降低失败率；而 `ProviderStrategy` 解析失败就直接抛异常
4. **和业务工具配合自然**：提交结果本身也是一次工具调用，和查询数据库、调用接口这些工具在同一个循环里完成。设置了 `ToolStrategy` 后，每次调用模型时都会要求它必须调用某个工具（`tool_choice="any"`，发给 OpenAI 时是 `"required"`），正常情况下模型不会直接用文字回答

**但它有一个前提：模型要遵守这个要求。** 本笔记用的中转模型就有不遵守的时候：测试 7.1 的例子时，不加系统提示词，模型查完通讯录后返回了一段普通文字（里面还混进了乱码），没有调用 `ContactInfo`。Agent 看到回复里没有工具调用，就直接结束了，`structured_response` 是 `None`。`handle_errors` 管不了这种情况，因为它只处理"调用了 `ContactInfo`，但参数校验失败"。所以实际使用时：

- 在系统提示词里写明"查到信息后调用某某工具提交结果"（7.1 的做法）
- 业务代码里检查 `result.get("structured_response")` 是不是 `None`，是的话记录下来或者重试

`ProviderStrategy` 也有适合的场景：确定使用 OpenAI 等支持原生结构化输出的模型，任务又是一步就能完成的抽取，并且希望由服务端保证格式（记得加 `strict=True`）。直接传类适合写原型、快速试验。

`ToolStrategy` 的 `handle_errors`、`tool_message_content`，以及用 `Union` 传入多个 schema 等用法，下一节（`06-Agent的ToolStrategy.ipynb`）细讲。

## 7.5 小结

1. 模型的 `with_structured_output` 调用一次就返回结构化结果；Agent 的 `response_format` 可以先调用工具，再把结果放进 `result["structured_response"]`
2. `response_format` 可以传：策略对象（`ToolStrategy` / `ProviderStrategy` / `AutoStrategy`）、类（Pydantic、dataclass、TypedDict）、JSON Schema 字典，或者 `None`。直接传类或字典，等同于 `AutoStrategy`
3. `SchemaT` 和 `ResponseT` 是分别定义的类型变量，用来联系策略配置与结构化结果的类型。`type[ContactInfo]` 描述类对象，`ToolStrategy[ContactInfo]` 描述对应的策略对象，`ContactInfo` 描述结果实例。泛型提供类型信息，LangChain 与 Pydantic 执行实际的数据解析和校验
4. `ProviderStrategy` 用厂商原生结构化输出，默认不开 strict，出错直接抛异常；`ToolStrategy` 把 schema 当成工具，兼容性好，还能自动纠错；自动选择依赖模型档案，可能选错
5. 实际业务中最常用的是显式写出的 `ToolStrategy`。使用时在系统提示词里写明何时提交结果，并检查 `structured_response` 是否为 `None`：模型不一定总会遵守"必须调用工具"的要求